In [ ]:
# Copyright (c) 2026 Nokia Bell Labs
# Licensed under the BSD 3 Clause license
# SPDX-License-Identifier: BSD-3-Clause

In [ ]:
import json
from collections import defaultdict
import matplotlib.pyplot as plt
import re
import pandas as pd
import os
import sys
import numpy as np
from matplotlib.colors import Normalize
import seaborn as sns


def analyze_scores(scores_dict, min_ari=0.98):
    results = {}

    for alg, step_dict in scores_dict.items():
        # Compute mean score over seeds for each step
        mean_scores = {}
        for step, seed_dict in step_dict.items():
            scores = list(seed_dict.values())
            mean_scores[step] = np.mean(scores)

        # Sort steps numerically
        sorted_steps = sorted(mean_scores.keys())
        sorted_means = [mean_scores[step] for step in sorted_steps]

        # Find the first step after which all mean scores are >= 0.95
        min_step = None
        for i, step in enumerate(sorted_steps):
            if all(m >= min_ari for m in sorted_means[i:]):
                min_step = step
                break

        # If such a step exists, compute mean and std of scores after that step
        if min_step is not None:
            filtered_scores = [
                mean_scores[step] for step in sorted_steps if step >= min_step
            ]
            mean_after = np.mean(filtered_scores)
            std_after = np.std(filtered_scores)
            results[alg] = {
                'min_step': min_step,
                'mean_after': mean_after,
                'std_after': std_after
            }
        else:
            results[alg] = {
                'min_step': None,
                'mean_after': None,
                'std_after': None
            }

    return results


# colors = ['blue', 'green', 'red', 'brown', 'black', 'orange', 'purple', 'pink', 'gray', 'lightcoral', 'darkred', 'magenta', 'navy', 'yellow']
colors = ['blue', 'blue', 'green', 'green', 'red', 'red', 'brown', 'brown', 'black', 'black', 'orange', 'orange', 'purple', 'purple']
markers = [".", "x", "s", "+", "*", "d", "o", "D", "^", "v", ">", "<", "p", "*",]
marker_and_color_cycler = plt.cycler('color', colors) + plt.cycler('marker', markers)
plt.rc(group='axes', prop_cycle=marker_and_color_cycler)
plt.rcParams['figure.edgecolor'] = 'black'
plt.rcParams['figure.figsize'] = (8,6)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 20
plt.rcParams['legend.fontsize'] = 20
plt.rcParams['errorbar.capsize'] = 3

def get_steps_and_scores_dict(item_key, data, match_re = r'alg=([^,]+), seed=(\d+)'):
    # Initialize dictionaries to store scores by alg, step, and seed
    scores_dict = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))  # alg -> step -> seed -> list of scores

    # Process each file in the data
    for filename, content in data.items():
        # Access the pairs data
        content = content.get('pairs', {})

        # Extract algorithm and seed values
        alg_seed_info = {}
        for key, value_list in content.items():
            if 'name' not in key:
                continue
            #print(value_list)
            for value in value_list:
                match = re.match(match_re, value)
                if match:
                    tmp1 = match.group(1)
                    tmp2 = match.group(2) if match.lastindex and match.lastindex >= 2 else None
                    if tmp2 is not None:
                        alg = tmp1
                        seed = int(tmp2)
                    else:
                        alg = 'CLoVE'
                        seed = int(tmp1)
                    alg_seed_info[filename] = (alg, seed)

        # Retrieve the algorithm and seed for this file
        alg, seed = alg_seed_info.get(filename, ("unknown", -1))

        # Process train data
        item_key_values = content.get(item_key, [])
        for line in item_key_values:
            step = int(re.search(r"step=(\d+)", line).group(1))
            score = float(line.split(",")[0])
            scores_dict[alg][step][seed].append(score)

    return scores_dict


def get_losses_steps_and_scores_dict(item_key, data):
    # Initialize a dictionary to store client losses by algorithm, step, and seed
    losses_dict = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))  # alg -> step -> seed -> list of losses

    # Process each file in the data
    for filename, content in data.items():
        # Access the pairs data
        pairs = content.get('pairs', {})

        # Extract algorithm and seed values
        alg_seed_info = {}
        for key, value_list in pairs.items():
            for value in value_list:
                match = re.match(r'alg=([^,]+), seed=(\d+)', value)
                if match:
                    alg = match.group(1)
                    seed = int(match.group(2))
                    alg_seed_info[filename] = (alg, seed)

        # Retrieve the algorithm and seed for this file
        alg, seed = alg_seed_info.get(filename, ("unknown", -1))

        # Extract client-specific losses
        for key, value_list in pairs.items():
            if key == item_key:
                for value in value_list:
                    match = re.match(r'([^,]+), step=(\d+)', value)
                    if match:
                        loss = float(match.group(1))
                        step = int(match.group(2))
                        losses_dict[alg][step][seed].append(loss)

    return losses_dict


def get_stats(scores_dict):
    # Compute averages, min, and max across seeds for each step
    scores_stat_dict = defaultdict(lambda: defaultdict(lambda: {"avg": 0, "min": float("inf"), "max": float("-inf")}))  # alg -> step -> stats
    for alg, steps in scores_dict.items():
        for step, seeds in steps.items():
            all_scores = [score for seed_scores in seeds.values() for score in seed_scores]
            if all_scores:
                avg_loss = sum(all_scores) / len(all_scores)
                min_loss = min(all_scores)
                max_loss = max(all_scores)
                scores_stat_dict[alg][step] = {"avg": avg_loss, "min": min_loss, "max": max_loss}

    return scores_stat_dict


def plot_alg_stats(stats, row_label):

    for alg, steps in stats.items():
        x = sorted(steps.keys())  # Steps on x-axis
        y = [steps[step]["avg"] for step in x]  # Average losses on y-axis
        yerr_lower = [steps[step]["avg"] - steps[step]["min"] for step in x]  # Lower error (avg - min)
        yerr_upper = [steps[step]["max"] - steps[step]["avg"] for step in x]  # Upper error (max - avg)
        yerr = [yerr_lower, yerr_upper]  # Combine lower and upper errors

        if alg == 'MA':
            alg = 'Model Averaging'
        if alg == 'GA':
            alg = 'Gradient Averaging'

        # plt.gca().set_prop_cycle(marker_and_color_cycler)
        unique_algs = set(stats.keys())     # If only one algorithm is included, do not include the algorithm's name in the labels.
        label = alg + row_label if len(unique_algs) != 1 else row_label
        line, = plt.plot(x, y, label=label)  # Line plot
        # line, = plt.plot(x, y, label=alg + row_label, color=colors[curve_index], marker=markers[curve_index])  # Line plot
        color = line.get_color()  # Use the line's color for error bars
        plt.errorbar(x, y, yerr=yerr, fmt='none', color=color, label='_nolegend_')  # Error bars with same color
        # plt.errorbar(x, y, yerr=yerr, fmt='o', color=colors[curve_index], label='_nolegend_')  # Error bars with same color
        x_human_friendly = np.array(x) + 1  # Steps on x-axis, added +1 for human friendliness
        plt.xticks(ticks=x, labels=x_human_friendly)


def get_init_type(row):
    ret_val = 's'
    if str(row['initial_model_parameters']) == 'different':
        ret_val = 'd'
    return ret_val


def get_avg_mode_type(row):
    if str(row['averaging_mode']) == 'model_averaging':
        ret_val = 'Model Averaging'
    elif str(row['averaging_mode']) == 'gradient_averaging':
        ret_val = 'Gradient Averaging'
    else:
        raise ValueError('Unknown averaging mode')
    return ret_val


def get_first_round_type(row):
    ret_val = 'r'
    if str(row['first_round_client_to_cluster_assignment']) == 'evaluation_based':
        ret_val = 'e'
    return ret_val


def get_row_md(row, type, sub_header=None):
    if type == 'Clustering accuracy':
        return ' (' + 'K=' + str(row['Clusters']) + ', ' + 'n=' + str(row['Datapoints per client']) + ')'
    elif type == 'Scaling with clients':
        return ' ('+ 'm=' + str(row['Clients per cluster']) +')'
    elif type == 'Clustering accuracy + loss over rounds':
        if sub_header == 'LINEAR':
            return ' (' + r'$\Delta$=' + str(row['Linear Delta']) +')'
        else:
            return ' (' + 'K=' + str(row['Clusters']) +')'
    elif type == 'Convergence speed':
        return ' (' + 'K=' + str(row['Clusters']) + ', ' + 'r=' + str(row['Total rounds']) +')'
    elif type == 'Initialization experiments':
        init_type = get_init_type(row)
        first_type = get_first_round_type(row)
        return ' (' + f'init:{init_type}'  ', ' + f'first:{first_type}' +')'
    elif type == 'MA vs GA':
        # avg_mode = get_avg_mode_type(row)
        # return avg_mode
        return ''
    return ''


def get_plt_legends(type, sub_header=None):
    if type == 'Clustering accuracy':
        return {
            'title': 'Clustering accuracy comparison among algorithms',
            'x':'Round',
            'y':'Adjusted Rand Index',
            # 'legend_title':'Algorithm'
            'legend_title':''
        }
    elif type == 'Scaling with clients':
        return {
            'title': 'Clustering accuracy: Scaling with clients',
            'x':'Round',
            'y':'Adjusted Rand Index',
            # 'legend_title':'Algorithm'
            'legend_title':''
        }
    elif type == 'Clustering accuracy + loss over rounds':
        return {
            'title': f'Clustering accuracy: {sub_header}',
            'x':'Round',
            'y':'Adjusted Rand Index',
            # 'legend_title':'Algorithm'
            'legend_title':''
        }
    elif type == 'Convergence speed':
        return {
            'title': 'Clustering accuracy: Convergence speed comparison',
            'x':'Round',
            'y':'Adjusted Rand Index',
            # 'legend_title':'Algorithm'
            'legend_title':''
        }
    elif type == 'Initialization experiments':
        return {
            'title': 'Clustering accuracy: Model initialization methods',
            'x':'Round',
            'y':'Adjusted Rand Index',
            # 'legend_title':'Algorithm'
            'legend_title':''
        }
    elif type == 'MA vs GA':
        return {
            'title': 'Clustering accuracy: MA vs GA',
            'x':'Round',
            'y':'Adjusted Rand Index',
            # 'legend_title':'Algorithm'
            'legend_title':''
        }

    return None


def parse_files_in_directory(directory):
    data = defaultdict(dict)  # Dictionary to hold the parsed data for all files

    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            filepath = os.path.join(directory, filename)

            with open(filepath, "r") as file:
                file_data = {"pairs": defaultdict(list), "json_data": None}
                lines = file.readlines()

                # Process all lines except the JSON part
                json_lines = []
                for line in lines:
                    line = line.strip()
                    if line.startswith("key="):
                        # Extract key and value while accounting for "=" and "," in the content
                        try:
                            prefix, key_value_pair = line.split("key=", 1)
                            key, value = key_value_pair.split(", value=", 1)
                            # Default dict auto handles initialization to empty list
                            file_data["pairs"][key].append(value)
                        except ValueError:
                            print(f"Skipping malformed line: {line}")
                    else:
                        # Collect all JSON lines
                        json_lines.append(line)

                # Combine JSON lines and parse them
                if json_lines:
                    try:
                        combined_json = "\n".join(json_lines)
                        file_data["json_data"] = json.loads(combined_json)
                    except json.JSONDecodeError:
                        print(f"Skipping invalid JSON in file: {filename}")

                data[filename] = file_data

    return data


def get_rev_map_dict(map_dict):
    rev_map = {}
    print(map_dict)
    for key, val in map_dict.items():
        for client_id in val:
            rev_map[client_id] = int(key)

    return rev_map


def reorder_columns(arr, indices):
    # Remove duplicates while preserving order
    seen = set()
    unique_indices = [i for i in indices if not (i in seen or seen.add(i))]

    # Append remaining indices that are not in unique_indices
    all_indices = unique_indices + [i for i in range(arr.shape[1]) if i not in unique_indices]

    return arr[:, all_indices]


def plot_loss_matrices(data_list, media_dir, exp_id, file_id_dict, include_cbar, include_ylabel):

    for d_item in data_list:
        print(d_item['id'])
        if str(d_item['id']) == str(exp_id):
            data_item = d_item
            break
    #    data_item = data_list[0]
    print(f"On experiment {data_item['id']}")
    data = data_item['results']
    row = data_item['meta_data']

    num_clusters = int(row['Clusters'])
    num_clients_per_cluster = int(row['Clients per cluster'])

    scale = file_id_dict["scale"]

    rows_to_keep = [i*num_clients_per_cluster for i in range(num_clusters)]

    file_name = file_id_dict['file_name']

    results_dict = {}
    # Process each file in the data
    for filename, content in data.items():
        # Access the pairs data
        loss_matrix_json = content.get('json_data', {})

        if filename not in file_id_dict.keys():
            continue
        print(filename)

        if 'loss_matrix_test' in loss_matrix_json and "client_to_selected_model_mapping_dict" in loss_matrix_json:
        # if 'loss_matrix_test' in loss_matrix_json and "selected_model_to_client_mapping_dict" in loss_matrix_json:
            loss_matrix = np.array(loss_matrix_json['loss_matrix_test'])
            last_matrix = loss_matrix[loss_matrix.shape[0] -1]
            #print(f"got loss matrix for {filename}, shape = {last_matrix.shape}")
            #print(last_matrix)

            # mapping_mat = np.array(loss_matrix_json['selected_model_to_client_mapping_dict'])
            # last_map_dict = mapping_mat[mapping_mat.shape[0] - 1]
            # print(f"got mapping matrix for {filename}")
            # print(last_map_dict)
            # rev_map = get_rev_map_dict(last_map_dict)
            # print(rev_map)
            # ordered_models_indx = [rev_map[rows_to_keep[i]] for i in range(len(rows_to_keep))]

            rev_mapping_mat = np.array(loss_matrix_json['client_to_selected_model_mapping_dict'])
            client_to_model_map = rev_mapping_mat[rev_mapping_mat.shape[0] - 1]
            print(f"client_to_model_map = {client_to_model_map}")
            ordered_models_indx = [client_to_model_map[str(row)] for row in rows_to_keep]
            print(f'Kept clients: {rows_to_keep}, corresponding models: {ordered_models_indx}')

            last_matrix = reorder_columns(last_matrix, ordered_models_indx)
            # print (last_matrix[:,1])
            # print (last_matrix[:,5])
            # print (last_matrix[:,7])

            filtered_matrix  = last_matrix[rows_to_keep]
            if file_id_dict[filename] == 'local':
                filtered_matrix = filtered_matrix[:, rows_to_keep]
            filtered_matrix = filtered_matrix * scale
            filtered_matrix = filtered_matrix.astype(int)
            results_dict[file_id_dict[filename]] = filtered_matrix

    number_of_models = [values.shape[1] for values in results_dict.values()]
    print(f"Number of models for each heatmap: {number_of_models}")
    if len(number_of_models) == 1:
        if number_of_models[0] == 1:
            figsize = (1,8)
        else:
            figsize = (8,8)
    else:
        figsize = (8,8) if any(x==1 for x in number_of_models) else (15,8)

    # Plot the heatmaps
    fig, axes = plt.subplots(1, 1, figsize=figsize, width_ratios=number_of_models)
    norm = Normalize(vmin=0.0, vmax=250)
    threshold = 200
    for key, data_matrix in results_dict.items():
        # Create a custom annotation array based on the threshold
        annot = np.where(data_matrix < threshold, data_matrix.astype(str), '')  # Only show values below threshold
        sns.heatmap(data_matrix, annot=annot, fmt="", cmap="coolwarm", cbar=False, norm=norm, ax=axes,
            # annot_kws={"size": 14, "fontname": "Arial"}
            annot_kws={"size": 14}
            )
        if key == "vanillaFL":
            heatmap_title = "FedAvg"
        elif key == "local":
            heatmap_title = "Local-only"
        else:
            heatmap_title = key
        axes.set_title(heatmap_title)
        axes.set_xlabel("Model ID")

    # Create a colorbar that spans both subplots, based on the values from both heatmaps
    # The first heatmap or the second can be used to generate the colorbar, as the color scales are the same
    if include_cbar is True:
        cax = fig.add_axes([axes.get_position().x1 + 0.5, axes.get_position().y0, 0.5, axes.get_position().height])
        cbar = fig.colorbar(axes.collections[0], cax=cax, orientation='vertical')
        cbar.set_label(f"Reconstruction Loss x {scale}")
    # fig.tight_layout()

    axes.set_ylabel("Cluster ID\nof clients from different clusters" if include_ylabel else "")
    #axes[0].set_ylabel("Cluster ID (MNIST Digit)", fontsize=14)
    #fig.text(0.5, 0.04, 'Model ID', ha='center', va='center', fontsize=14)  # X-axis label
    #plot_file = os.path.join(plot_file, algo_type + '.pdf')
    plot_file = os.path.join(media_dir, file_name)
    plt.savefig(plot_file + '.pdf', format='pdf', bbox_inches='tight')
    plt.savefig(plot_file + '.png', format='png', bbox_inches='tight')
    plt.show()


class ExperimentPlotting:
    def __init__(self, base_path, results_dir='results', exp_drv_dir='experiment_driver', drvr_file_name='experiment_list_unsupervised_extended.csv'):
        self.df = None
        self.base_path = base_path
        full_input_file_path = os.path.join(self.base_path, exp_drv_dir, drvr_file_name)

        if not os.path.exists(full_input_file_path):
            print("There is no driver file so exiting out")
            sys.exit(1)
        else:
            self.df = pd.read_csv(full_input_file_path)

        self.results_dir = os.path.join(self.base_path, results_dir)
        self.media_dir = os.path.join(self.base_path, 'media')
        # Make sure you have media dir under base_path
        if not os.path.exists(self.media_dir):
            os.makedirs(self.media_dir)
            print(f"Directory '{self.media_dir}' created.")

    def do_clustering_plot(self, value='Clustering accuracy', sub_selector=None, plt_ylim=None, key='adjusted_rand_score_train', header='Experiment type', num_yticks=5, doing_MASSIVE_analysis=False):
        data_list = []
        try:
            if self.df is not None:
                for index, row in self.df.iterrows():
                    if row.get(header, None) == value:

                        # Used for Clustering accuracy + loss over rounds experiments
                        if sub_selector is not None:
                            sub_header = sub_selector['header']
                            sub_value = sub_selector['value']
                            if row.get(sub_header, None) != sub_value:
                                continue

                        # Extract the Experiment ID list for the current row
                        experiment_id = str(row['Experiment ID'])
                        result_path = os.path.join(self.results_dir, experiment_id)
                        result_data = parse_files_in_directory(result_path)
                        data_list.append({'id': experiment_id, 'results': result_data, 'meta_data': row})
            else:
                print("Data has not been loaded yet. Please 'read_file' first.")
                return
        except ValueError:
            print("Error parsing file.")
            return

        ################## ADDED FOR MASSIVE #################################################################
        if value == 'Clustering accuracy MASSIVE':
            value = 'Clustering accuracy'
            doing_MASSIVE_analysis=True
        ################## ADDED FOR MASSIVE #################################################################

        # Mapping of experiment description to desired figure name
        value_to_fig_name = {
            'Clustering accuracy': 'clustering_accuracy',
            'Clustering accuracy + loss over rounds': 'clustering_accuracy',
            'Scaling with clients': 'client_scaling',
            'Convergence speed': 'convergence_speed',
            'MA vs GA': 'MA_vs_GA',
            'Initialization experiments': 'initialization_experiments',
        }

        if sub_selector is not None:
            plot_file = os.path.join(self.media_dir, value_to_fig_name[value] + '_' + sub_selector['value'])
        else:
            plot_file = os.path.join(self.media_dir, value_to_fig_name[value])

        # Plotting with error bars
        plt.figure()

        for plt_row in data_list:
            if value == 'MA vs GA':
                scores = get_steps_and_scores_dict(key, plt_row['results'], match_re = r'am=([^,]+), seed=(\d+)')
            elif value == 'Clustering accuracy + loss over rounds':
                scores = get_steps_and_scores_dict(key, plt_row['results'], match_re = r'seed=(\d+)')
            else:
                scores = get_steps_and_scores_dict(key, plt_row['results'])

            ################## ADDED FOR MASSIVE #################################################################
            if doing_MASSIVE_analysis:
                MASSIVE_results = analyze_scores(scores, min_ari=0.9)
                print(MASSIVE_results)
            ################## ADDED FOR MASSIVE #################################################################

            stats = get_stats(scores)

            # Used for Clustering accuracy + loss over rounds experiments
            if sub_selector is not None:
                row_label = get_row_md(plt_row['meta_data'], value, sub_header=sub_value)
            else:
                row_label = get_row_md(plt_row['meta_data'], value)

            # Override to get for simpler scaling graph with CLoVE only, not IFCA
            if value == 'Scaling with clients':
                if 'IFCA' in stats.keys():
                    del stats['IFCA']
                    num_clusters = int(row['Clusters'])
                    clients_per_cluster = int(row_label.strip()[3:-1])
                    row_label = f"M = {num_clusters * clients_per_cluster}"

            plot_alg_stats(stats, row_label)
            if (value == 'Convergence speed') or ((value == 'Clustering accuracy + loss over rounds') and (sub_value == 'FEMNIST')):
                # Fix the x-axis tick labels for the graph with more rounds by showing half the ticks and labels
                xticks = plt.gca().get_xticks()
                xticklabels = [label.get_text() for label in plt.gca().get_xticklabels()]
                plt.xticks(ticks=xticks[::2], labels=xticklabels[::2])

        # Used for Clustering accuracy + loss over rounds experiments
        if sub_selector is not None:
            all_legends = get_plt_legends(value, sub_header=sub_value)
        else:
            all_legends = get_plt_legends(value)

        if all_legends is None:
            return

        # plt.title(all_legends['title'], fontsize=14)
        plt.xlabel(all_legends['x'])
        plt.ylabel(all_legends['y'])
        # plt.legend(title=all_legends['legend_title'], fontsize=12, title_fontsize=14)
        plt.legend(title=all_legends['legend_title'], loc='best')

        margin = 0.05
        if plt_ylim is None:
            y_min = 0.0
            y_max = 1.0
        else:
            y_min = plt_ylim[0]
            y_max = plt_ylim[1]

        plt.ylim(y_min - margin, y_max + margin)

        # Set custom ticks starting at y_min
        tick_values = np.linspace(y_min, y_max, num=num_yticks)  # Adjust number of ticks as needed
        plt.yticks(tick_values)

        # Set grid
        plt.grid(True)
        # # Set background transparent
        # plt.gcf().patch.set_alpha(0)
        plt.tight_layout()

        plt.savefig(plot_file + '.pdf', format='pdf', bbox_inches='tight')
        plt.savefig(plot_file + '.png', format='png', bbox_inches='tight')
        plt.show()


    def do_loss_matrices(self, value='Clustering accuracy', exp_id=401, file_id_dict=None, header='Experiment type', include_cbar=True, include_ylabel=True):
        data_list = []
        try:
            if self.df is not None:
                for index, row in self.df.iterrows():
                    if row.get(header, None) == value:
                        # Extract the Experiment ID list for the current row
                        experiment_id = str(row['Experiment ID'])
                        result_path = os.path.join(self.results_dir, experiment_id)
                        result_data = parse_files_in_directory(result_path)
                        data_list.append({'id': experiment_id, 'results': result_data, 'meta_data': row})
            else:
                print("Data has not been loaded yet. Please 'read_file' first.")
                return
        except ValueError:
            print("Error parsing file.")
            return

        plot_loss_matrices(data_list=data_list, media_dir=self.media_dir, exp_id=exp_id, file_id_dict=file_id_dict, include_cbar=include_cbar, include_ylabel=include_ylabel)


In [ ]:
# # MASSIVE (July 2025) experiments:  Unsupervised MNIST with larger # of clients: experiment IDs 2101, 2102, 2103. 2104
# ################## ADDED FOR MASSIVE #################################################################
#
# directory_path = "../"
# plotter = ExperimentPlotting(directory_path, results_dir='results/runs_unsupervised_extended')
# #MNIST Clustering Accuracy
# plotter.do_clustering_plot(value='Clustering accuracy MASSIVE', sub_selector= {'header': 'Dataset', 'value': 'MNIST'}, doing_MASSIVE_analysis=True)


In [ ]:
###############################################################################################################
###############################################################################################################
###############################################################################################################

# Generate Plots

directory_path = "../"
plotter = ExperimentPlotting(directory_path, results_dir='results/runs_unsupervised_extended')

# MNIST Clustering accuracy
# plotter.do_clustering_plot(value='Clustering accuracy', sub_selector= {'header': 'Dataset', 'value': 'MNIST'}, plt_ylim=[-0.05, 1], num_yticks=6)

# Scaling with clients
plotter.do_clustering_plot(value='Scaling with clients')

# MA vs GA
plotter.do_clustering_plot(value='MA vs GA')

# Convergence Speed
# plotter.do_clustering_plot(value='Convergence speed')

# FEMNIST Clustering accuracy
# plotter.do_clustering_plot(value='Clustering accuracy + loss over rounds', sub_selector= {'header': 'Dataset', 'value': 'FEMNIST'})

#LINEAR Clustering Accuracy
plotter.do_clustering_plot(value='Clustering accuracy + loss over rounds', sub_selector= {'header': 'Dataset', 'value': 'LINEAR'})

# Initialization experiments
plotter.do_clustering_plot(value='Initialization experiments')


#parsed_data = parse_files_in_directory(directory_path)
#get_steps_and_scores_dict('rand_score_train', parsed_data)
#plot_average_adjusted_rand_scores(parsed_data)
#compare_client_losses_across_algorithms(parsed_data, 10)

# Not needed
#plot_adjusted_rand_scores(parsed_data)

# Print the parsed data
# for file, content in parsed_data.items():
#     print(f"File: {file}")
#     print("Key-Value Pairs:")
#     for k, v in content["pairs"].items():
#         print(f"  {k}: {v}")
#     print("JSON Data:", content["json_data"])
#     print()

In [ ]:
###############################################################################################################
###############################################################################################################
###############################################################################################################

# Generate Loss Heatmaps for MNIST

directory_path = "../"
plotter = ExperimentPlotting(directory_path, results_dir='results/runs_unsupervised_extended')

run_dict = {
    401: {'exp_id': 401, 'scale': 1000},
    403: {'exp_id': 403, 'scale': 10000},
    404: {'exp_id': 404, 'scale': 50000},
    405: {'exp_id': 405, 'scale': 5000}
}

run_ids = {
    'MNIST': 401,
    # 'FEMNIST': 403,
    # 'UNSW': 404,
    # 'LINEAR': 405,
}

for dataset, run_id in run_ids.items():
    file_id_dict = {
        'alg=CLoVE, seed=1.txt': 'CLoVE',
        # 'alg=IFCA, seed=1.txt': 'IFCA',
        'file_name': "loss_heatmap_CLoVE_on_" + dataset,
        'scale': run_dict[run_id]["scale"]
    }
    plotter.do_loss_matrices(value='Loss over rounds', exp_id=run_id, file_id_dict=file_id_dict, include_cbar=False, include_ylabel=True)

    file_id_dict = {
        # 'alg=CLoVE, seed=1.txt': 'CLoVE',
        'alg=IFCA, seed=1.txt': 'IFCA',
        'file_name': "loss_heatmap_IFCA_on_" + dataset,
        'scale': run_dict[run_id]["scale"]
    }
    plotter.do_loss_matrices(value='Loss over rounds', exp_id=run_id, file_id_dict=file_id_dict, include_cbar=False, include_ylabel=False)

    file_id_dict = {
        'alg=local, seed=1.txt': 'local',
        # 'alg=vanillaFL, seed=1.txt': 'vanillaFL',
        'file_name': "loss_heatmap_local_only_on_" + dataset,
        'scale': run_dict[run_id]["scale"]
    }
    plotter.do_loss_matrices(value='Loss over rounds', exp_id=run_id, file_id_dict=file_id_dict, include_cbar=False, include_ylabel=False)

    file_id_dict = {
        # 'alg=local, seed=1.txt': 'local',
        'alg=vanillaFL, seed=1.txt': 'vanillaFL',
        'file_name': "loss_heatmap_FedAvg_on_" + dataset,
        'scale': run_dict[run_id]["scale"]
    }
    plotter.do_loss_matrices(value='Loss over rounds', exp_id=run_id, file_id_dict=file_id_dict, include_cbar=True, include_ylabel=False)


# file_id_dict = {
#     'alg=CLoVE, seed=1.txt': 'CLoVE',
#     'alg=IFCA, seed=1.txt': 'IFCA',
#     'alg=local, seed=1.txt': 'local',
#     'alg=vanillaFL, seed=1.txt': 'vanillaFL',
#     'file_name': "loss_heatmaps_all"
# }
# plotter.do_loss_matrices(value='Loss over rounds', exp_id=401, file_id_dict=file_id_dict)